# Pipeline from a 1D spectrum

Simulate a BSPEC rate map from a BT-Settl template, build a `RampModel`, sync BSPEC calibrations from KOA if needed, then run `Stage0Pipeline` → `Stage1Pipeline` → `Stage2Pipeline` with defaults and show the stellar-RV preview.

**Environment variables** (set these in your shell / notebook kernel before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache directory (and SQLite DB) |
| `KECK_OBSERVER_EMAIL` | Syncing / downloading from KOA | Keck Observer account email |
| `KECK_OBSERVER_PASSWORD` | Syncing / downloading from KOA | Keck Observer account password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.  
The KOA cell below syncs metadata from remote KOA and downloads any **BSPEC** calibration files that are not already in the cache.


In [ ]:
import os

# OUTPUT DIRECTORY — update this to your desired output directory
OUT = "/Users/ckilpatrick/Downloads"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


## 1. Sync / verify KOA caldb (BSPEC)


In [ ]:
from hispecdrp.calibrations import verify_koa_calibrations

# Requires env vars: KOA_CALIBRATION_CACHE, KECK_OBSERVER_EMAIL, KECK_OBSERVER_PASSWORD
SPECTROGRAPH = "BSPEC"
summary = verify_koa_calibrations(SPECTROGRAPH)

# Print out the number of records in the caldb and the number of files that were downloaded
print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])


## 2. Simulated rate map from a 1D spectrum → `RampModel`

Load a BT-Settl template, create a rate map with `SimulateRateMap`, then build a raw UTR `RampModel` and provide the caldb files that were used to create the physical ramp.

In [ ]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

FIBERS = [1, 2, 3]  # SPECTROGRAPH is set in the KOA cell above

# Build a rate map from a 1D BT-Settl template and a raw UTR RampModel (KOA masters from cache).
ramp, cals, rate_map = create_ramp_from_spectrum(
    OUT,
    spectrograph=SPECTROGRAPH,
    fibers=FIBERS,
    template_filename="lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5",
    mag_ab=10.0,
    filter_band="J",  # BSPEC (yJ)
)
print(type(ramp).__name__, ramp.meta.filename, "shape", getattr(ramp, "data", None).shape if hasattr(ramp, "data") else None)
print("Stage-0 cals:", sorted(cals))
print("rate_map shape:", getattr(rate_map, "shape", None))


## 3. Stage 0 → 1 → 2 via pipeline classes

Primitives resolve calibrations from the KOA caldb using the ramp / L0 / L1 metadata.

In [ ]:
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline

l0 = Stage0Pipeline(output_dir=OUT).apply(ramp)

l1 = Stage1Pipeline(output_dir=OUT).apply(l0)

l2 = Stage2Pipeline(output_dir=OUT).apply(l1)

print("L0:", type(l0).__name__, getattr(l0.meta, "filename", None))
print("L1:", type(l1).__name__, getattr(l1.meta, "filename", None))
print("L2:", type(l2).__name__, getattr(l2.meta, "filename", None))
